## Import Library

In [1]:
import csv
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Scraping data

In [2]:
results = []
target_count = 200

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/122.0.0.0 Safari/537.36'
}urls = [
    'https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops',
    'https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops?page=2',
    'https://webscraper.io/test-sites/e-commerce/allinone/computers/tablets',
    'https://webscraper.io/test-sites/e-commerce/allinone/phones/touch',
    'https://webscraper.io/test-sites/e-commerce/allinone/phones/touch?page=2',
]

for url in urls:
    if len(results) >= target_count:
        break

    response = requests.get(url, headers=headers)
    print(f"Mengakses: {url} | Status Code: {response.status_code}")

    if response.status_code != 200:
        continue

    soup = BeautifulSoup(response.text, 'html.parser')

    products = soup.find_all('div', class_='thumbnail')
    print(f"Jumlah produk ditemukan di halaman ini: {len(products)}")

    for prod in products:
        if len(results) >= target_count:
            break

        title_elem = prod.find('a', class_='title')
        nama_produk = title_elem.get_text(strip=True) if title_elem else "Unknown"

        # Tentukan Kategori Berdasarkan Nama/URL
        if "Laptop" in url or "Laptop" in nama_produk:
            kategori = "Laptop"
        elif "Tablet" in url or "Tablet" in nama_produk:
            kategori = "Tablet"
        else:
            kategori = "Phone"

        # Atribut Angka 1: Harga (Format teks awal, misal "$299.99")
        price_elem = prod.find('h4', class_='price')
        harga_teks = price_elem.get_text(strip=True) if price_elem else "$0"

        # Atribut Angka 2: Jumlah Review (Angka)
        review_elem = prod.find('p', class_='ratings')
        review_count_elem = review_elem.find('span', class_='float-end') if review_elem else None
        review_teks = review_count_elem.get_text(strip=True) if review_count_elem else "0"

        results.append({
            'nama_produk': nama_produk,
            'harga_teks': harga_teks,
            'jumlah_review_teks': review_teks,
            'kategori': kategori
        })

print(f"\nTotal data berhasil terkumpul: {len(results)} item (Target >= 200)")

# Langkah 6: Tampilkan hasilnya dengan perulangan for (Menampilkan 5 sampel data pertama)
print("\n--- Sampel 5 Data Pertama Hasil Scraping ---")
for i, item in enumerate(results[:5], 1):
    print(f"{i}. {item}")

Mengakses: https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops | Status Code: 200
Jumlah produk ditemukan di halaman ini: 117
Mengakses: https://webscraper.io/test-sites/e-commerce/allinone/computers/laptops?page=2 | Status Code: 200
Jumlah produk ditemukan di halaman ini: 117

Total data berhasil terkumpul: 200 item (Target >= 200)

--- Sampel 5 Data Pertama Hasil Scraping ---
1. {'nama_produk': 'Asus VivoBook...', 'harga_teks': '$295.99', 'jumlah_review_teks': '0', 'kategori': 'Phone'}
2. {'nama_produk': 'Prestigio Smar...', 'harga_teks': '$299', 'jumlah_review_teks': '0', 'kategori': 'Phone'}
3. {'nama_produk': 'Prestigio Smar...', 'harga_teks': '$299', 'jumlah_review_teks': '0', 'kategori': 'Phone'}
4. {'nama_produk': 'Aspire E1-510', 'harga_teks': '$306.99', 'jumlah_review_teks': '0', 'kategori': 'Phone'}
5. {'nama_produk': 'Lenovo V110-15...', 'harga_teks': '$321.94', 'jumlah_review_teks': '0', 'kategori': 'Phone'}


In [5]:
csv_filename = "Laprak_Scraping.csv"

fieldnames = ["nama_produk", "harga_teks", "jumlah_review_teks", "kategori"]

with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"Data berhasil disimpan ke file CSV: {csv_filename}")

Data berhasil disimpan ke file CSV: Laprak_Scraping.csv


In [7]:
df = pd.read_csv("Laprak_Scraping.csv")

print("--- Tipe Data Awal (Sebelum Konversi) ---")
print(df.dtypes)
print("\n")

# Langkah 9: Pastikan kolom angka bertipe angka (int/float)
# Membersihkan simbol '$' pada harga_teks lalu mengubahnya menjadi float
df["harga"] = (
    df["harga_teks"].str.replace(r"[^\d.]", "", regex=True).astype(float)
)

# Memastikan jumlah_review_teks (stok_simulasi) bertipe integer
df["jumlah_review_teks"] = df["jumlah_review_teks"].astype(int)

# Menata ulang kolom sesuai syarat (Atribut: harga [angka], jumlah_review_teks [angka], kategori [kategori])
df = df[["nama_produk", "harga", "jumlah_review_teks", "kategori"]]

# Menyimpan hasil akhir yang bersih untuk Praktikum 3 & 4
clean_filename = "data_praktikum_clean.csv"
df.to_csv(clean_filename, index=False)

print("--- Tipe Data Setelah Konversi (Cleaned) ---")
print(df.dtypes)
print(f"\nData bersih berhasil disimpan ke: {clean_filename}")

print("\n--- Preview 10 Data Teratas (Siap untuk Praktikum 3 & 4) ---")
display(df.head(10))

--- Tipe Data Awal (Sebelum Konversi) ---
nama_produk           object
harga_teks            object
jumlah_review_teks     int64
kategori              object
dtype: object


--- Tipe Data Setelah Konversi (Cleaned) ---
nama_produk            object
harga                 float64
jumlah_review_teks      int64
kategori               object
dtype: object

Data bersih berhasil disimpan ke: data_praktikum_clean.csv

--- Preview 10 Data Teratas (Siap untuk Praktikum 3 & 4) ---


,nama_produk,harga,jumlah_review_teks,kategori
0,Asus VivoBook...,295.99,0,Phone
1,Prestigio Smar...,299.00,0,Phone
2,Prestigio Smar...,299.00,0,Phone
3,Aspire E1-510,306.99,0,Phone
4,Lenovo V110-15...,321.94,0,Phone
5,Lenovo V110-15...,356.49,0,Phone
6,Hewlett Packar...,364.46,0,Phone
7,Acer Aspire 3...,372.70,0,Phone
8,Acer Aspire A3...,379.94,0,Phone
9,Acer Aspire ES...,379.95,0,Phone
